# UABR — Phase 5: interference pilot (does plain SFT break the CALL/ANSWER policy?)

**Kaggle settings:** Accelerator **GPU T4 x2** · Internet **On** · **+ Add Input → Your Work**: the Phase 2 notebook (`data_v1`) and the Phase 4 notebook (`delegation_v1.pt`).

**What happens:**
1. Two Standard SFT runs (seeds 11 and 29) from `delegation_v1` on Dolly + Alpaca, no replay, saving 25% / 50% / 100% checkpoints.
2. Every checkpoint is scored on all dev prompts: routing AUROC, accuracy at a 10% call budget (re-picked and frozen threshold), harmful answers, mutual information, instruction loss.
3. The preregistered rule in `docs/pilot_decision.md` prints **GO**, **RECALIBRATION_ONLY** or **NO_CHANGE**.

`QUICK = True` → Run All (~10 min) checks the pipeline. Then `QUICK = False` → **Save Version › Save & Run All (Commit)** (~1 h).

In [ ]:
QUICK = True
REPO_URL = "https://github.com/GTX-Pradeep/uabr-slm.git"

import os, glob, json, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

data_hits = glob.glob("/kaggle/input/**/data_v1/manifest.json", recursive=True)
deleg_hits = [p for p in glob.glob("/kaggle/input/**/delegation_v1.pt", recursive=True) if "delegation_quick" not in p]
assert data_hits, "data_v1 not found: Add Input -> Your Work -> the Phase 2 notebook"
assert deleg_hits, "delegation_v1.pt (full run) not found: Add Input -> Your Work -> the Phase 4 notebook"
DATA, DELEG = os.path.dirname(data_hits[0]), deleg_hits[0]
SEEDS = [11, 29]
RUNS = [f"{WORK}/runs/sft_standard_s{s}" + ("_quick" if QUICK else "") for s in SEEDS]
PILOT = f"{WORK}/runs/pilot" + ("_quick" if QUICK else "")
print("DATA  :", DATA); print("DELEG :", DELEG); print("RUNS  :", RUNS); print("PILOT :", PILOT)
!git log --oneline -1
!python -m pytest tests -q -x

## 1. Standard SFT, two seeds (~10–15 min each in full mode)
Same instruction data and settings for both seeds; only the batch order differs. Checkpoints at 25% and 50% plus `final.pt` (100%).

In [ ]:
quick = "--quick" if QUICK else ""
for seed, run in zip(SEEDS, RUNS):
    !rm -rf {run}
    !python scripts/run_sft.py --init {DELEG} --data {DATA} --seed {seed} --out {run} --max_minutes 300 {quick}

## 2. Score every checkpoint on dev + apply the decision rule (~15–25 min in full mode)
`delegation_v1` is the 0% point. Its thresholds for 5 / 10 / 20% call rates are frozen and reused, to separate real damage (ranking) from a calibration shift (threshold).

In [ ]:
!python scripts/pilot_eval.py --data {DATA} --delegation {DELEG} --runs {",".join(RUNS)} --out {PILOT} {quick}

## 3. Plots

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
t = pd.read_csv(f"{PILOT}/pilot_table.csv")
base = t[t.seed == "all"].iloc[0]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for seed in SEEDS:
    s = pd.concat([base.to_frame().T, t[t.seed == str(seed)]]).astype({"frac": float})
    for a, col, title in zip(ax, ["auroc", "acc_budget_10", "acc_frozen_thr_10"],
                             ["Routing AUROC", "Accuracy @10% (re-picked threshold)", "Accuracy @10% (frozen threshold)"]):
        a.plot(s.frac * 100, s[col].astype(float), "o-", label=f"seed {seed}")
        a.set_title(title); a.set_xlabel("% of SFT done"); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(f"{PILOT}/pilot_curves.png", dpi=120); plt.show()
print(json.dumps(json.load(open(f"{PILOT}/pilot_report.json"))["decision"]))

## 4. Before Phase 6
- Copy the per-seed differences and the decision into `docs/pilot_decision.md` and get all four sign-offs.
- The full-run `final.pt` of each seed is the **Standard SFT** branch of Phase 6 (attach this notebook's output there).